In [2]:
pip install --upgrade transformers


In [3]:
!pip install datasets

In [4]:
from datasets import load_dataset
dataset = load_dataset("iSemantics/conllpp-ner-ar")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/843 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/738k [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/184k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/167k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10250 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2383 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2572 [00:00<?, ? examples/s]

In [5]:
import re

def clean_data(data):
  data =re.sub(r'[^\w\s]','',data)
  data =re.sub(r'\s+',' ',data).strip()
  return data

In [6]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("aubmindlab/bert-base-arabertv02")



tokenizer_config.json:   0%|          | 0.00/381 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/825k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.64M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [7]:
def clean_invalid_tags(dataset):
    # Define label_list here, before it's used in the filter function
    label_list = dataset["train"].features["ner_tags"].feature.names

    for split in  ['train', 'validation', 'test']:  # تحقق في كل تقسيم
        print(f"Cleaning {split} dataset...")
        # تصفية البيانات
        dataset[split] = dataset[split].filter(lambda example: all(tag < len(label_list) for tag in example["ner_tags"]))
        # طباعة عدد الأمثلة بعد التصفية
        print(f"Number of examples in {split} after cleaning: {len(dataset[split])}")

# تنظيف البيانات
clean_invalid_tags(dataset)

Cleaning train dataset...


Filter:   0%|          | 0/10250 [00:00<?, ? examples/s]

Number of examples in train after cleaning: 10250
Cleaning validation dataset...


Filter:   0%|          | 0/2383 [00:00<?, ? examples/s]

Number of examples in validation after cleaning: 2383
Cleaning test dataset...


Filter:   0%|          | 0/2572 [00:00<?, ? examples/s]

Number of examples in test after cleaning: 2572


entity labeling using (BIO Scheme)

In [8]:
label_list =dataset["train"].features["ner_tags"].feature.names
#label2id ={label: i for i,label in enumerate(label_list)} ما نحتاجها هنا

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], padding="max_length", truncation=True, max_length=128, is_split_into_words=True)
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(label[word_idx] if label[word_idx] != -100 else -100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs


In [9]:
tokenized_train = dataset["train"].map(tokenize_and_align_labels, batched=True)
tokenized_val = dataset["validation"].map(tokenize_and_align_labels, batched=True)
tokenized_test = dataset["test"].map(tokenize_and_align_labels, batched=True)


Map:   0%|          | 0/10250 [00:00<?, ? examples/s]

Map:   0%|          | 0/2383 [00:00<?, ? examples/s]

Map:   0%|          | 0/2572 [00:00<?, ? examples/s]

Label Mapping


In [10]:
id2label = {idx: label for idx, label in enumerate(label_list)}
label2id = {label: idx for idx, label in enumerate(label_list)}

Import and load the Pre-trained Model

In [11]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "aubmindlab/bert-base-arabertv02",  # Pre-trained AraBERT model
    num_labels=len(label_list),        # Number of unique NER tags
    id2label=id2label,                 # Map from ID to label
    label2id=label2id                  # Map from label to ID
)

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at aubmindlab/bert-base-arabertv02 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# Traning

In [12]:
%env CUDA_LAUNCH_BLOCKING=1


env: CUDA_LAUNCH_BLOCKING=1


In [13]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(64000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

In [14]:
# Import necessary tools from Hugging Face Transformers and scikit-learn
from transformers import Trainer, TrainingArguments, AutoModelForTokenClassification, DataCollatorForTokenClassification
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import numpy as np

# Create a data collator for token classification (NER task)
# This ensures input batches are padded and formatted correctly
data_collator = DataCollatorForTokenClassification(tokenizer)

# Define a function to compute evaluation metrics (accuracy, precision, recall, F1)
def compute_metrics(p):
    predictions, labels = p
    # Select the most likely class for each token
    predictions = np.argmax(predictions, axis=2)

    # Remove padding tokens (with label -100) from evaluation
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    # Compute evaluation metrics using flattened label lists
    precision, recall, f1, _ = precision_recall_fscore_support(
        sum(true_labels, []),
        sum(true_predictions, []),
        average="macro"  # Calculate macro-average across all classes
    )
    acc = accuracy_score(sum(true_labels, []), sum(true_predictions, []))  # Compute overall accuracy
    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
    }

# Define the list of possible labels for NER
label_list = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

# Create dictionaries to map between label IDs and label names (required by the model)
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

# Load the pre-trained AraBERT model and configure it for token classification (NER)
model = AutoModelForTokenClassification.from_pretrained(
    "aubmindlab/bert-base-arabertv02",  # Pretrained AraBERT model
    num_labels=len(label_list),         # Number of NER classes
    id2label=id2label,                  # ID-to-label mapping
    label2id=label2id                   # Label-to-ID mapping
)

# Set up training arguments (number of epochs, batch size, logging, etc.)
training_args = TrainingArguments(
    output_dir="./results",              # Where to save model checkpoints and outputs
    per_device_train_batch_size=16,      # Training batch size
    per_device_eval_batch_size=16,       # Evaluation batch size
    num_train_epochs=5,                  # Total number of training epochs
    weight_decay=0.01,                   # Weight decay to avoid overfitting
    logging_dir="./logs",                # Directory for training logs
    logging_steps=10,                    # Log every 10 steps
    save_steps=500,                      # Save model every 500 steps
    do_eval=True,                        # Run evaluation during training
    do_train=True,                       # Enable training
    fp16=True,                           # Enable mixed-precision training (16-bit floats)
)

# Initialize the Trainer class that handles training and evaluation
trainer = Trainer(
    model=model,                         # The model to train
    args=training_args,                  # Training arguments
    train_dataset=tokenized_train,       # Preprocessed training dataset
    eval_dataset=tokenized_val,          # Preprocessed validation dataset
    tokenizer=tokenizer,                 # Tokenizer used for preprocessing
    data_collator=data_collator,         # Data collator for token classification
    compute_metrics=compute_metrics,     # Evaluation function
)

# Start the training process
trainer.train()


Some weights of BertForTokenClassification were not initialized from the model checkpoint at aubmindlab/bert-base-arabertv02 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-14-d812a255f379>:71: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
10,1.442200
20,0.666500
30,0.538200
40,0.497500
50,0.343800
60,0.415100
70,0.400900
80,0.376200
90,0.300100
100,0.319300


TrainOutput(global_step=3205, training_loss=0.11065829224221978, metrics={'train_runtime': 1145.854, 'train_samples_per_second': 44.726, 'train_steps_per_second': 2.797, 'total_flos': 3348076570560000.0, 'train_loss': 0.11065829224221978, 'epoch': 5.0})

In [15]:
from sklearn.metrics import classification_report
import numpy as np

# Evaluate the model on the validation set
eval_results = trainer.evaluate(tokenized_val)

# Print the basic evaluation metrics
print("\n" + "="*50)
print("Final Evaluation Results on Validation Set:")
print("="*50)
print(f"{'Loss:':<20} {eval_results['eval_loss']:.4f}")
print(f"{'Accuracy:':<20} {eval_results['eval_accuracy']:.4f}")
print(f"{'F1 Score:':<20} {eval_results['eval_f1']:.4f}")
print(f"{'Precision:':<20} {eval_results['eval_precision']:.4f}")
print(f"{'Recall:':<20} {eval_results['eval_recall']:.4f}")

# Function to generate a classification report
def get_classification_report(predictions, labels):
    # Remove padding tokens (where label = -100)
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    # Flatten the lists
    flat_predictions = [item for sublist in true_predictions for item in sublist]
    flat_labels = [item for sublist in true_labels for item in sublist]

    # Generate the classification report
    report = classification_report(flat_labels, flat_predictions, output_dict=True)
    return report, flat_labels, flat_predictions

# Get predictions from the model
predictions, labels, _ = trainer.predict(tokenized_val)
predictions = np.argmax(predictions, axis=2)

# Get the detailed report
detailed_report, true_labels, pred_labels = get_classification_report(predictions, labels)

# Print the detailed classification report
print("\n" + "="*50)
print("Detailed Classification Report:")
print("="*50)
print(classification_report(true_labels, pred_labels))

# Print performance analysis per class
print("\n" + "="*50)
print("Per-Class Performance Analysis:")
print("="*50)
for label in label_list:
    if label in detailed_report:  # Skip averages
        print(f"\nPerformance for class {label}:")
        print(f"- Precision: {detailed_report[label]['precision']:.3f}")
        print(f"- Recall: {detailed_report[label]['recall']:.3f}")
        print(f"- F1 Score: {detailed_report[label]['f1-score']:.3f}")
        print(f"- Support: {detailed_report[label]['support']}")



Final Evaluation Results on Validation Set:
Loss:                0.2123
Accuracy:            0.9544
F1 Score:            0.8543
Precision:           0.8718
Recall:              0.8410

Detailed Classification Report:
              precision    recall  f1-score   support

       B-LOC       0.88      0.88      0.88      1646
      B-MISC       0.78      0.72      0.75       684
       B-ORG       0.91      0.89      0.90      3013
       B-PER       0.90      0.95      0.93      2585
       I-LOC       0.77      0.78      0.78       233
      I-MISC       0.81      0.60      0.69       357
       I-ORG       0.87      0.81      0.84      1574
       I-PER       0.95      0.95      0.95      2381
           O       0.98      0.98      0.98     31273

    accuracy                           0.95     43746
   macro avg       0.87      0.84      0.85     43746
weighted avg       0.95      0.95      0.95     43746


Per-Class Performance Analysis:

Performance for class O:
- Precision: 0.978

# **Testing and Save The Model**

In [16]:
# Evaluate the model on the test set
test_results = trainer.evaluate(tokenized_test)

# Print evaluation metrics on the test set
print("\n" + "="*50)
print("Final Evaluation Results on Test Set:")
print("="*50)
print(f"{'Loss:':<20} {test_results['eval_loss']:.4f}")
print(f"{'Accuracy:':<20} {test_results['eval_accuracy']:.4f}")
print(f"{'F1 Score:':<20} {test_results['eval_f1']:.4f}")
print(f"{'Precision:':<20} {test_results['eval_precision']:.4f}")
print(f"{'Recall:':<20} {test_results['eval_recall']:.4f}")

# Get predictions on the test set
test_predictions, test_labels, _ = trainer.predict(tokenized_test)
test_predictions = np.argmax(test_predictions, axis=2)

# Generate detailed report
test_report, true_test_labels, pred_test_labels = get_classification_report(test_predictions, test_labels)

# Print the detailed classification report
print("\n" + "="*50)
print("Detailed Classification Report (Test Set):")
print("="*50)
print(classification_report(true_test_labels, pred_test_labels))

# Per-Class Performance Analysis
print("\n" + "="*50)
print("Per-Class Performance Analysis (Test Set):")
print("="*50)
for label in label_list:
    if label in test_report:
        print(f"\nPerformance for class {label}:")
        print(f"- Precision: {test_report[label]['precision']:.3f}")
        print(f"- Recall: {test_report[label]['recall']:.3f}")
        print(f"- F1 Score: {test_report[label]['f1-score']:.3f}")
        print(f"- Support: {test_report[label]['support']}")



Final Evaluation Results on Test Set:
Loss:                0.2957
Accuracy:            0.9435
F1 Score:            0.8241
Precision:           0.8229
Recall:              0.8256

Detailed Classification Report (Test Set):
              precision    recall  f1-score   support

       B-LOC       0.87      0.87      0.87      1722
      B-MISC       0.65      0.69      0.67       601
       B-ORG       0.88      0.86      0.87      3231
       B-PER       0.91      0.94      0.93      2317
       I-LOC       0.77      0.76      0.77       321
      I-MISC       0.58      0.59      0.58       284
       I-ORG       0.83      0.80      0.81      1836
       I-PER       0.94      0.95      0.95      2093
           O       0.98      0.98      0.98     28447

    accuracy                           0.94     40852
   macro avg       0.82      0.83      0.82     40852
weighted avg       0.94      0.94      0.94     40852


Per-Class Performance Analysis (Test Set):

Performance for class O:
- 

In [17]:
# Save the trained model and tokenizer
model_path = "./saved_model_arabert_ner"
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

print(f"\nModel and tokenizer saved to {model_path}")



Model and tokenizer saved to ./saved_model_arabert_ner


# **Deployment**

In [24]:
# Import the necessary libraries
from transformers import AutoModelForTokenClassification, AutoTokenizer, pipeline

# Load the model and tokenizer from the local directory
model = AutoModelForTokenClassification.from_pretrained("saved_model_arabert_ner") # Corrected path to the saved model
tokenizer = AutoTokenizer.from_pretrained("saved_model_arabert_ner") # Corrected path to the saved model


In [29]:

# Create the NER pipeline
ner_pipeline = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

# Test sentence
sentence = "أعلن الملك سلمان بن عبدالعزيز عن دعم المملكة العربية السعودية لاتفاقية السلام في الشرق الأوسط خلال القمة العربية في القاهرة."

# Run prediction
ner_results = ner_pipeline(sentence)

# Display results
print("\n" + "="*50)
print("Named Entities Extracted:")
print("="*50)
for entity in ner_results:
    print(f"Entity: {entity['word']:<15} | Label: {entity['entity_group']:<10} | Score: {entity['score']:.2f}")


Device set to use cuda:0



Named Entities Extracted:
Entity: سلمان بن عبدالعزيز | Label: PER        | Score: 0.76
Entity: المملكة العربية السعودية | Label: LOC        | Score: 0.50
Entity: الشرق الأوسط    | Label: LOC        | Score: 0.89
Entity: العربية         | Label: MISC       | Score: 0.95
Entity: القاهرة         | Label: LOC        | Score: 0.98
